# Quantum Portfolio Optimization

## QUBO Formulation for 8-Asset Binary Portfolio Selection

**Objective:**

min_x  q * sum_{ij} x_i x_j σ_{ij}  -  sum_i x_i μ_i  +  λ(sum_i x_i - B)²

where x_i ∈ {0,1}, B = 4 (select exactly 4 of 8 assets).

After expanding and using x_i² = x_i for binary variables, this becomes x^T Q x where:
- Q_ii = q·σ_ii + λ(1 - 2B) - μ_i
- Q_ij = q·σ_ij + λ   (i ≠ j)

## 1. Load Data and Select Top 8 Assets

In [1]:
import numpy as np
import pandas as pd
import cvxpy as cp
from itertools import product

# --- Load data ---
assets       = pd.read_csv('investment_dataset_assets.csv')
cov_df       = pd.read_csv('investment_dataset_full(covariance).csv', index_col=0)

mu    = assets['exp_return'].values
Sigma = cov_df.values.astype(float)
w_min = assets['w_min'].values.astype(float)
w_max = assets['w_max'].values.astype(float)
n     = len(mu)

# --- Continuous Markowitz to find top 8 assets ---
w = cp.Variable(n)
q_base = 1.0
prob = cp.Problem(
    cp.Minimize(q_base * cp.quad_form(w, Sigma) - mu @ w),
    [cp.sum(w) == 1, w >= w_min, w <= w_max]
)
prob.solve()
continuous_weights = w.value

assets['continuous_weight'] = continuous_weights
top8 = assets.nlargest(8, 'continuous_weight').reset_index(drop=True)
rep_ids = [assets.index[assets['asset_id'] == aid].item() for aid in top8['asset_id']]

mu_8    = mu[rep_ids]
Sigma_8 = Sigma[np.ix_(rep_ids, rep_ids)]

print("Top 8 assets by continuous Markowitz weight:")
print(top8[['asset_id', 'sector', 'exp_return', 'volatility', 'continuous_weight']].to_string(
    index=False, float_format='{:.4f}'.format))

Top 8 assets by continuous Markowitz weight:
asset_id        sector  exp_return  volatility  continuous_weight
    A001 Equities Intl      0.0845      0.1993             0.0800
    A004   Equities US      0.0823      0.1729             0.0800
    A005 Equities Intl      0.0860      0.1956             0.0800
    A016 Equities Intl      0.0879      0.2016             0.0800
    A030   Equities US      0.0803      0.1760             0.0800
    A023 Equities Intl      0.0835      0.2017             0.0757
    A049   Equities US      0.0807      0.1759             0.0716
    A006     HY Credit      0.0737      0.1205             0.0600


## 2. Build the QUBO Matrix

In [2]:
B = 4       # number of assets to select
q = 1.0     # risk aversion

# lambda should be large enough to enforce the constraint
# but not so large it drowns out the objective.
# Heuristic: scale of the largest eigenvalue of the cost matrix.
eigvals = np.linalg.eigvalsh(q * Sigma_8)
lambda_ = float(np.max(np.abs(eigvals)))
print(f"lambda = {lambda_:.6f}  (max eigenvalue of q*Sigma_8)")

# --- Build Q matrix ---
Q = q * Sigma_8.copy()                            # risk: q * σ_ij for all i,j
Q += lambda_ * np.ones((8, 8))                    # penalty cross-terms: +λ for all pairs
diag_correction = lambda_ * (1 - 2*B) - mu_8      # penalty diagonal + return term
np.fill_diagonal(Q, np.diag(Q) + diag_correction) # fix diagonal entries

print(f"\nQ matrix shape: {Q.shape}")
print(f"Q diagonal (first 4): {np.diag(Q)[:4].round(6)}")
print(f"Q off-diag sample Q[0,1]: {Q[0,1]:.6f}")

lambda = 0.099675  (max eigenvalue of q*Sigma_8)

Q matrix shape: (8, 8)
Q diagonal (first 4): [-0.642842 -0.650465 -0.645848 -0.645297]
Q off-diag sample Q[0,1]: 0.102370


## 3. Verify QUBO Against Brute-Force

Evaluate x^T Q x for all C(8,4) = 70 binary vectors with exactly 4 ones.
The minimum should match the classical brute-force result.

In [3]:
# --- Brute-force: evaluate x^T Q x for all bitstrings ---
best_obj  = np.inf
best_bits = None
all_results = []

for bits in product([0, 1], repeat=8):
    x = np.array(bits, dtype=float)
    k = x.sum()
    obj = x @ Q @ x

    if k == B:  # feasible solution
        # Also compute the original portfolio metrics for comparison
        w_eq = x / k
        port_return = mu_8 @ w_eq
        port_vol    = np.sqrt(w_eq @ Sigma_8 @ w_eq)
        sharpe      = port_return / port_vol

        all_results.append({
            'bits': bits,
            'qubo_obj': obj,
            'return': port_return,
            'volatility': port_vol,
            'sharpe': sharpe,
        })

        if obj < best_obj:
            best_obj  = obj
            best_bits = bits

# Sort by QUBO objective
all_results_df = pd.DataFrame(all_results).sort_values('qubo_obj')

print("Top 5 portfolios by QUBO objective (lower = better):")
print("-" * 80)
for _, row in all_results_df.head(5).iterrows():
    selected = [top8.iloc[i]['asset_id'] for i, b in enumerate(row['bits']) if b == 1]
    print(f"  obj={row['qubo_obj']:.6f}  return={row['return']*100:.2f}%  "
          f"vol={row['volatility']*100:.2f}%  Sharpe={row['sharpe']:.3f}")
    print(f"    Assets: {', '.join(selected)}")

print(f"\nBest QUBO bitstring: {best_bits}")
selected = [top8.iloc[i]['asset_id'] for i, b in enumerate(best_bits) if b == 1]
print(f"Selected assets: {', '.join(selected)}")

# --- Verify: check that infeasible solutions (k ≠ 4) have higher objectives ---
infeasible_objs = []
for bits in product([0, 1], repeat=8):
    x = np.array(bits, dtype=float)
    if x.sum() != B and x.sum() > 0:
        infeasible_objs.append(x @ Q @ x)

print(f"\nFeasible best objective:    {best_obj:.6f}")
print(f"Infeasible min objective:   {min(infeasible_objs):.6f}")
print(f"Penalty working: {best_obj < min(infeasible_objs)}")

Top 5 portfolios by QUBO objective (lower = better):
--------------------------------------------------------------------------------
  obj=-1.359030  return=8.06%  vol=9.98%  Sharpe=0.807
    Assets: A004, A005, A030, A006
  obj=-1.355728  return=8.11%  vol=10.14%  Sharpe=0.799
    Assets: A004, A016, A030, A006
  obj=-1.353460  return=8.21%  vol=10.34%  Sharpe=0.794
    Assets: A001, A004, A016, A006
  obj=-1.352773  return=8.11%  vol=10.24%  Sharpe=0.792
    Assets: A004, A016, A049, A006
  obj=-1.350992  return=8.02%  vol=10.19%  Sharpe=0.788
    Assets: A001, A004, A030, A006

Best QUBO bitstring: (0, 1, 1, 0, 1, 0, 0, 1)
Selected assets: A004, A005, A030, A006

Feasible best objective:    -1.359030
Infeasible min objective:   -1.346438
Penalty working: True


## 4. Convert QUBO to Ising Hamiltonian

Substitute x_i = (1 - Z_i) / 2 to get H = sum J_ij Z_i Z_j + sum h_i Z_i + const.
This is the form QAOA operates on.

In [4]:
# --- QUBO to Ising conversion ---
# Substitute x_i = (1 - z_i) / 2, z_i in {+1, -1}
#
# x^T Q x = sum_i Q_ii * x_i  +  2 * sum_{i<j} Q_ij * x_i * x_j   (using x_i^2 = x_i)
#
# Expanding x_i = (1-z_i)/2 and collecting terms:
#   const = sum_i Q_ii/2  +  (1/4) * sum_{i!=j} Q_ij
#   h_i   = -Q_ii/2  -  (1/2) * sum_{j!=i} Q_ij
#   J_ij  = Q_ij/4   (off-diagonal, used in full symmetric z @ J @ z)

Q_diag   = np.diag(Q)
Q_offdiag = Q.copy()
np.fill_diagonal(Q_offdiag, 0)

J     = Q_offdiag / 4.0
h     = -Q_diag / 2.0 - Q_offdiag.sum(axis=1) / 2.0
const =  Q_diag.sum() / 2.0 + Q_offdiag.sum() / 4.0

print("Ising coefficients:")
print(f"  h (local fields): {h.round(6)}")
print(f"  J range: [{J[J!=0].min():.6f}, {J[J!=0].max():.6f}]")
print(f"  const: {const:.6f}")

# --- Verify: Ising energy must exactly match QUBO energy for every bitstring ---
print("\nVerification (top 5 feasible solutions):")
all_match = True
for _, row in all_results_df.head(5).iterrows():
    x = np.array(row['bits'], dtype=float)
    z = 1 - 2 * x                        # x_i=1 -> z_i=-1, x_i=0 -> z_i=+1
    ising_energy = const + h @ z + z @ J @ z
    qubo_energy  = row['qubo_obj']
    match = np.isclose(qubo_energy, ising_energy)
    all_match &= match
    print(f"  QUBO={qubo_energy:.6f}  Ising={ising_energy:.6f}  match={match}")

print(f"\nAll verified: {all_match}")

Ising coefficients:
  h (local fields): [-0.061984 -0.045994 -0.060964 -0.056846 -0.046998 -0.062658 -0.051208
 -0.025373]
  J range: [0.025122, 0.030713]
  const: -1.088425

Verification (top 5 feasible solutions):
  QUBO=-1.359030  Ising=-1.359030  match=True
  QUBO=-1.355728  Ising=-1.355728  match=True
  QUBO=-1.353460  Ising=-1.353460  match=True
  QUBO=-1.352773  Ising=-1.352773  match=True
  QUBO=-1.350992  Ising=-1.350992  match=True

All verified: True


## 5. QAOA Circuit (p=1) on Bloqade

One QAOA layer built as a squin kernel via closure — gamma and beta are baked in at creation time.

Circuit structure:
  1. H gate on every qubit to initialise |+>^8
  2. Cost unitary: for each pair (i,j) apply Rzz(2*gamma*J_ij) = CNOT - Rz - CNOT; then Rz(2*gamma*h_i) per qubit
  3. Mixer: Rx(2*beta) on every qubit
  4. Measure all 8 qubits

In [5]:
from bloqade import squin
from bloqade.pyqrack import StackMemorySimulator

N_QUBITS = 8

def make_qaoa_circuit(gamma_list, beta_list, J: np.ndarray, h: np.ndarray):
    """Return a squin kernel for p-layer QAOA. gamma_list and beta_list have length p."""
    p = len(gamma_list)
    assert len(beta_list) == p

    # Pre-compute ZZ pair indices and angles per layer — flat lists, no tuple unpacking
    all_zz_i, all_zz_j = [], []
    for i in range(N_QUBITS):
        for j in range(i + 1, N_QUBITS):
            if abs(J[i, j]) > 1e-12:
                all_zz_i.append(i)
                all_zz_j.append(j)
    n_pairs = len(all_zz_i)

    # All angles per layer, as flat lists of floats
    zz_angles_per_layer = [
        [float(2 * gamma_list[l] * J[all_zz_i[k], all_zz_j[k]]) for k in range(n_pairs)]
        for l in range(p)
    ]
    z_angles_per_layer = [
        [float(2 * gamma_list[l] * h[i]) for i in range(N_QUBITS)]
        for l in range(p)
    ]
    x_angles = [float(2 * beta_list[l]) for l in range(p)]

    @squin.kernel
    def qaoa_circuit():
        qubits = squin.qalloc(N_QUBITS)

        # 1. Initialise |+>^8
        for i in range(N_QUBITS):
            squin.h(qubits[i])

        # 2. p QAOA layers
        for l in range(p):
            # Cost unitary — ZZ couplings via CNOT-Rz-CNOT
            for k in range(n_pairs):
                squin.cx(qubits[all_zz_i[k]], qubits[all_zz_j[k]])
                squin.rz(zz_angles_per_layer[l][k], qubits[all_zz_j[k]])
                squin.cx(qubits[all_zz_i[k]], qubits[all_zz_j[k]])

            # Cost unitary — local Z fields
            for i in range(N_QUBITS):
                squin.rz(z_angles_per_layer[l][i], qubits[i])

            # Mixer
            for i in range(N_QUBITS):
                squin.rx(x_angles[l], qubits[i])

        return squin.broadcast.measure(qubits)

    return qaoa_circuit


def estimate_energy(gamma_list, beta_list, J, h, Q, shots=300):
    """Run the QAOA circuit and return mean QUBO energy over FEASIBLE shots only (k=B)."""
    if not isinstance(gamma_list, list):
        gamma_list = [gamma_list]
        beta_list  = [beta_list]

    circuit = make_qaoa_circuit(gamma_list, beta_list, J, h)
    sim     = StackMemorySimulator(min_qubits=N_QUBITS)
    task    = sim.task(circuit)
    results = task.batch_run(shots=shots)

    total, feasible_count = 0.0, 0
    for shot in results:
        x = np.array([int(b) for b in shot], dtype=float)
        if int(x.sum()) == B:          # only count feasible (k=B) shots
            total += x @ Q @ x
            feasible_count += 1

    if feasible_count == 0:
        return 0.0, 0.0                # no feasible shots
    return total / feasible_count, feasible_count / shots


# Quick smoke-test
gamma_test, beta_test = 0.5, 0.3
E_test, feas = estimate_energy(gamma_test, beta_test, J, h, Q, shots=200)
print(f"Smoke-test energy at gamma={gamma_test}, beta={beta_test}: {E_test:.6f}  (feasibility: {feas*100:.1f}%)")
print(f"Brute-force optimum:                                        {best_obj:.6f}")

Smoke-test energy at gamma=0.5, beta=0.3: -1.299127  (feasibility: 18.5%)
Brute-force optimum:                                        -1.359030


## 6. Optimise QAOA Parameters (Grid Search)

Sweep gamma in [0, pi] and beta in [0, pi/2] to find the parameters that minimise expected energy.
The QAOA circuit is periodic in both parameters with these periods, so this covers the full landscape.

In [6]:
GRID_SIZE = 5
SHOTS     = 100

gamma_vals = np.linspace(0.1, 2.0, GRID_SIZE)
beta_vals  = np.linspace(0.1, 1.5, GRID_SIZE)

print(f"p=1 grid search ({GRID_SIZE**2} circuits × {SHOTS} shots)...")
energy_grid = np.full((GRID_SIZE, GRID_SIZE), np.inf)
feas_grid   = np.zeros((GRID_SIZE, GRID_SIZE))

for gi, gamma in enumerate(gamma_vals):
    for bi, beta in enumerate(beta_vals):
        E, feas = estimate_energy([gamma], [beta], J, h, Q, shots=SHOTS)
        if feas > 0:
            energy_grid[gi, bi] = E
        feas_grid[gi, bi] = feas
    print(f"  row {gi+1}/{GRID_SIZE}  gamma={gamma:.2f}  "
          f"avg_feasibility={feas_grid[gi].mean()*100:.1f}%")

best_idx   = np.unravel_index(np.argmin(energy_grid), energy_grid.shape)
best_gamma = gamma_vals[best_idx[0]]
best_beta  = beta_vals[best_idx[1]]
best_E     = energy_grid[best_idx]

best_gammas = [best_gamma]
best_betas  = [best_beta]
best_p      = 1

print(f"\nBest: gamma={best_gamma:.3f}  beta={best_beta:.3f}")
print(f"Best feasible energy: {best_E:.6f}  (brute-force: {best_obj:.6f})")
print(f"Approx ratio:         {best_E / best_obj:.4f}  (1.0 = perfect)")
print(f"Avg feasibility:      {feas_grid.mean()*100:.1f}%  "
      f"(random baseline: {100*70/256:.1f}%)")

p=1 grid search (25 circuits × 100 shots)...
  row 1/5  gamma=0.10  avg_feasibility=24.2%
  row 2/5  gamma=0.57  avg_feasibility=23.8%
  row 3/5  gamma=1.05  avg_feasibility=21.0%
  row 4/5  gamma=1.52  avg_feasibility=20.6%
  row 5/5  gamma=2.00  avg_feasibility=21.6%

Best: gamma=1.525  beta=0.800
Best feasible energy: -1.311317  (brute-force: -1.359030)
Approx ratio:         0.9649  (1.0 = perfect)
Avg feasibility:      22.2%  (random baseline: 27.3%)


## 7. Sample Optimal Circuit and Evaluate Results

Run many shots at the best (gamma, beta) and find how often the quantum solver returns the correct bitstring.

In [7]:
FINAL_SHOTS = 1000

circuit = make_qaoa_circuit(best_gammas, best_betas, J, h)
sim     = StackMemorySimulator(min_qubits=N_QUBITS)
task    = sim.task(circuit)
results = task.batch_run(shots=FINAL_SHOTS)

from collections import Counter

bitstring_counts = Counter()
for shot in results:
    bits = tuple(int(b) for b in shot)
    bitstring_counts[bits] += 1

shot_results = []
for bits, count in bitstring_counts.items():
    x = np.array(bits, dtype=float)
    k = int(x.sum())
    if k == 0:
        continue
    obj     = x @ Q @ x
    w_eq    = x / k
    port_return = float(mu_8 @ w_eq)
    port_vol    = float(np.sqrt(w_eq @ Sigma_8 @ w_eq))
    shot_results.append({
        'bits': bits,
        'count': count,
        'freq': count / FINAL_SHOTS,
        'k': k,
        'qubo_obj': obj,
        'return': port_return,
        'volatility': port_vol,
        'sharpe': port_return / port_vol if port_vol > 0 else 0,
        'is_optimal': bits == best_bits,
        'feasible': k == B,
    })

shot_df    = pd.DataFrame(shot_results).sort_values('qubo_obj')
feasible_df = shot_df[shot_df['feasible']]

feasible_rate = shot_df['feasible'].sum() / FINAL_SHOTS * shot_df['count'].sum() / len(shot_df)
feasible_shots = sum(r['count'] for r in shot_results if r['feasible'])

print(f"p={best_p} QAOA — final results over {FINAL_SHOTS} shots")
print(f"Feasible (k={B}) shots: {feasible_shots} / {FINAL_SHOTS} ({100*feasible_shots/FINAL_SHOTS:.1f}%)")
print(f"Optimal bitstring {best_bits} found: {bitstring_counts[best_bits]} times "
      f"({100*bitstring_counts[best_bits]/FINAL_SHOTS:.1f}%)")

print(f"\nTop 5 most frequent feasible bitstrings:")
print("-" * 80)
top_feasible = feasible_df.sort_values('count', ascending=False).head(5)
for _, row in top_feasible.iterrows():
    selected = [top8.iloc[i]['asset_id'] for i, b in enumerate(row['bits']) if b == 1]
    marker = " <-- OPTIMAL" if row['is_optimal'] else ""
    print(f"  freq={row['freq']*100:.1f}%  return={row['return']*100:.2f}%  "
          f"Sharpe={row['sharpe']:.3f}  obj={row['qubo_obj']:.4f}{marker}")
    print(f"    Assets: {', '.join(selected)}")

print(f"\nSummary:")
print(f"  Brute-force optimal:  return={all_results_df.iloc[0]['return']*100:.2f}%  "
      f"Sharpe={all_results_df.iloc[0]['sharpe']:.3f}")
if not feasible_df.empty:
    best_qaoa_row = feasible_df.iloc[0]
    selected = [top8.iloc[i]['asset_id'] for i, b in enumerate(best_qaoa_row['bits']) if b == 1]
    print(f"  Best QAOA feasible:   return={best_qaoa_row['return']*100:.2f}%  "
          f"Sharpe={best_qaoa_row['sharpe']:.3f}  (Assets: {', '.join(selected)})")

p=1 QAOA — final results over 1000 shots
Feasible (k=4) shots: 65 / 1000 (6.5%)
Optimal bitstring (0, 1, 1, 0, 1, 0, 0, 1) found: 1 times (0.1%)

Top 5 most frequent feasible bitstrings:
--------------------------------------------------------------------------------
  freq=0.1%  return=8.06%  Sharpe=0.807  obj=-1.3590 <-- OPTIMAL
    Assets: A004, A005, A030, A006
  freq=0.1%  return=7.93%  Sharpe=0.686  obj=-1.2994
    Assets: A004, A030, A049, A006
  freq=0.1%  return=8.42%  Sharpe=0.684  obj=-1.2907
    Assets: A004, A005, A016, A049
  freq=0.1%  return=8.27%  Sharpe=0.680  obj=-1.2902
    Assets: A005, A030, A023, A049
  freq=0.1%  return=8.37%  Sharpe=0.682  obj=-1.2897
    Assets: A005, A016, A030, A049

Summary:
  Brute-force optimal:  return=8.06%  Sharpe=0.807
  Best QAOA feasible:   return=8.06%  Sharpe=0.807  (Assets: A004, A005, A030, A006)
